# 01 - 词嵌入与 Word2Vec


词嵌入是自然语言处理的第一道门槛：模型不能直接理解文字，必须先把词变成向量。本 notebook 在原 Python 教程基础上，把 One-Hot、分布假说、Skip-gram、负采样、`nn.Embedding` 等概念拆开讲清楚。

学习目标：

- 理解为什么 One-Hot 无法表达语义关系。
- 理解词嵌入为什么是低维、稠密、可学习的表示。
- 知道 Word2Vec 的 Skip-gram 训练目标：用中心词预测上下文。
- 看懂负采样的直觉：拉近真实上下文，推远随机噪声词。
- 能把 `nn.Embedding` 看成一个可训练查找表。

## 环境准备与导入

这一段保留原教程的导入、标题打印和基础设置。先运行它，后续代码单元会复用这里导入的库、函数和随机种子。

In [1]:
"""
第五章 5.1：词嵌入与 Word2Vec
==============================

在 NLP 中，第一个问题是：如何让计算机"理解"文字？

本节内容：
1. One-Hot 编码的问题
2. 词嵌入的直觉
3. Word2Vec 原理
4. 从零实现 Skip-gram
5. 词向量的有趣性质
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 60)
print("第五章 5.1：词嵌入与 Word2Vec")
print("=" * 60)

第五章 5.1：词嵌入与 Word2Vec


## 1. One-Hot 的问题


One-Hot 的优点是简单：每个词都有唯一编号，不会混淆。但它的缺点也正来自这种“唯一性”：所有不同词之间都正交，余弦相似度为 0。

这意味着模型无法从向量本身知道“猫”和“狗”更接近，“猫”和“苹果”更远。真实 NLP 模型需要的是语义空间：相似的词在空间里距离更近，不相似的词距离更远。

In [2]:
print("\n" + "=" * 60)
print("1. One-Hot 编码的问题")
print("=" * 60)

print("""
【One-Hot 编码回顾】
假设词表有 5 个词: [猫, 狗, 鱼, 苹果, 香蕉]

  猫   = [1, 0, 0, 0, 0]
  狗   = [0, 1, 0, 0, 0]
  鱼   = [0, 0, 1, 0, 0]
  苹果 = [0, 0, 0, 1, 0]
  香蕉 = [0, 0, 0, 0, 1]

【问题】
1. 维度灾难：实际词表有 10 万+ 个词
   每个词是 100000 维的向量！太大了

2. 稀疏：向量中只有 1 个位置是 1，其余全是 0
   浪费存储和计算

3. 无语义关系：
   cos(猫, 狗) = 0
   cos(猫, 苹果) = 0
   → 所有词的距离都一样！猫和狗不比猫和苹果更"近"

【我们需要什么？】
  一种低维、稠密的表示，能捕捉词的语义关系：
  
  猫 ≈ [0.8, -0.2, 0.5, ...]  (比如 128 维)
  狗 ≈ [0.7, -0.3, 0.4, ...]  (和猫接近！因为都是动物)
  苹果 ≈ [-0.1, 0.6, -0.3, ...] (和猫很远，因为不是动物)
  
  这就是词嵌入 (Word Embedding)！
""")

# 演示 One-Hot 的问题
vocab = ['猫', '狗', '鱼', '苹果', '香蕉']
one_hot = np.eye(len(vocab))

print("One-Hot 编码:")
for word, vec in zip(vocab, one_hot):
    print(f"  {word}: {vec}")

# 计算余弦相似度
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f"\n余弦相似度:")
print(f"  cos(猫, 狗) = {cosine_similarity(one_hot[0], one_hot[1]):.4f}")
print(f"  cos(猫, 苹果) = {cosine_similarity(one_hot[0], one_hot[3]):.4f}")
print(f"  → 所有词对之间的相似度都是 0！无法区分语义关系")


1. One-Hot 编码的问题

【One-Hot 编码回顾】
假设词表有 5 个词: [猫, 狗, 鱼, 苹果, 香蕉]

  猫   = [1, 0, 0, 0, 0]
  狗   = [0, 1, 0, 0, 0]
  鱼   = [0, 0, 1, 0, 0]
  苹果 = [0, 0, 0, 1, 0]
  香蕉 = [0, 0, 0, 0, 1]

【问题】
1. 维度灾难：实际词表有 10 万+ 个词
   每个词是 100000 维的向量！太大了

2. 稀疏：向量中只有 1 个位置是 1，其余全是 0
   浪费存储和计算

3. 无语义关系：
   cos(猫, 狗) = 0
   cos(猫, 苹果) = 0
   → 所有词的距离都一样！猫和狗不比猫和苹果更"近"

【我们需要什么？】
  一种低维、稠密的表示，能捕捉词的语义关系：

  猫 ≈ [0.8, -0.2, 0.5, ...]  (比如 128 维)
  狗 ≈ [0.7, -0.3, 0.4, ...]  (和猫接近！因为都是动物)
  苹果 ≈ [-0.1, 0.6, -0.3, ...] (和猫很远，因为不是动物)

  这就是词嵌入 (Word Embedding)！

One-Hot 编码:
  猫: [1. 0. 0. 0. 0.]
  狗: [0. 1. 0. 0. 0.]
  鱼: [0. 0. 1. 0. 0.]
  苹果: [0. 0. 0. 1. 0.]
  香蕉: [0. 0. 0. 0. 1.]

余弦相似度:
  cos(猫, 狗) = 0.0000
  cos(猫, 苹果) = 0.0000
  → 所有词对之间的相似度都是 0！无法区分语义关系


## 2. 词嵌入的直觉


词嵌入的核心来自分布假说：一个词的含义由它经常出现的上下文决定。猫和狗经常出现在“养了一只”“会叫”“宠物”等相似上下文中，所以模型会把它们学得更近。

初学时可以把词嵌入想成“自动学习出来的特征表”。每一维不一定有明确人工含义，但整体向量能编码词的语义、语法和上下文习惯。

In [3]:
print("\n" + "=" * 60)
print("2. 词嵌入的直觉")
print("=" * 60)

print("""
【词嵌入 (Word Embedding)】
将每个词映射到一个低维稠密向量（如 100-300 维），
使得语义相近的词，向量也相近。

【如何学习词嵌入？】
核心思想（分布假说）：
  "You shall know a word by the company it keeps."
  一个词的含义由它周围的词决定。

例如：
  "我养了一只__，它会喵喵叫" → 猫
  "我养了一只__，它会汪汪叫" → 狗
  
  因为"猫"和"狗"经常出现在相似的上下文中，
  所以它们的向量应该很接近。

而 "苹果" 出现在 "我吃了一个__" 这样的上下文中，
和动物的上下文很不同，所以向量应该很远。

【Word2Vec 的两种模式】
1. Skip-gram: 给定中心词，预测周围的词
   输入: "猫" → 预测: "养", "一只", "它", "喵喵叫"
   
2. CBOW: 给定周围的词，预测中心词
   输入: "养", "一只", "它", "喵喵叫" → 预测: "猫"
""")


2. 词嵌入的直觉

【词嵌入 (Word Embedding)】
将每个词映射到一个低维稠密向量（如 100-300 维），
使得语义相近的词，向量也相近。

【如何学习词嵌入？】
核心思想（分布假说）：
  "You shall know a word by the company it keeps."
  一个词的含义由它周围的词决定。

例如：
  "我养了一只__，它会喵喵叫" → 猫
  "我养了一只__，它会汪汪叫" → 狗

  因为"猫"和"狗"经常出现在相似的上下文中，
  所以它们的向量应该很接近。

而 "苹果" 出现在 "我吃了一个__" 这样的上下文中，
和动物的上下文很不同，所以向量应该很远。

【Word2Vec 的两种模式】
1. Skip-gram: 给定中心词，预测周围的词
   输入: "猫" → 预测: "养", "一只", "它", "喵喵叫"

2. CBOW: 给定周围的词，预测中心词
   输入: "养", "一只", "它", "喵喵叫" → 预测: "猫"



## 3. 从零实现 Skip-gram


Skip-gram 的训练数据是一批 `(中心词, 上下文词)` 对。模型看到中心词后，要给真实上下文词打高分，给随机负样本词打低分。

这里的两个嵌入矩阵分别表示“作为中心词时的向量”和“作为上下文词时的向量”。实际使用时，我们通常取中心词嵌入作为最终词向量。

In [4]:
print("\n" + "=" * 60)
print("3. 从零实现 Skip-gram Word2Vec")
print("=" * 60)

# 简单语料
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "the dog chased the cat",
    "the cat and the dog are friends",
]

# 构建词表
def build_vocab(corpus):
    words = set()
    for sentence in corpus:
        for word in sentence.split():
            words.add(word)
    word2idx = {w: i for i, w in enumerate(sorted(words))}
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(corpus)
vocab_size = len(word2idx)
print(f"词表: {word2idx}")
print(f"词表大小: {vocab_size}")

# 生成 Skip-gram 训练数据
def generate_skipgram_data(corpus, word2idx, window_size=2):
    """生成 (中心词, 上下文词) 对"""
    pairs = []
    for sentence in corpus:
        words = sentence.split()
        for i, word in enumerate(words):
            center_idx = word2idx[word]
            # 窗口内的上下文词
            for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
                if j != i:
                    context_idx = word2idx[words[j]]
                    pairs.append((center_idx, context_idx))
    return pairs

pairs = generate_skipgram_data(corpus, word2idx, window_size=2)
print(f"\n训练样本数: {len(pairs)}")
print(f"前5个样本:")
for center, context in pairs[:5]:
    print(f"  ({idx2word[center]}, {idx2word[context]})")

# 实现 Skip-gram 模型
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # 两个嵌入矩阵：中心词和上下文词
        self.center_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embed_dim)
    
    def forward(self, center_words, context_words):
        # 获取嵌入向量
        center_embeds = self.center_embeddings(center_words)   # (batch, embed_dim)
        context_embeds = self.context_embeddings(context_words) # (batch, embed_dim)
        
        # 计算得分（点积）
        scores = (center_embeds * context_embeds).sum(dim=1)   # (batch,)
        return scores

# 训练
embed_dim = 10  # 小词表用小维度就够
model = SkipGram(vocab_size, embed_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 准备数据
center_words = torch.LongTensor([p[0] for p in pairs])
context_words = torch.LongTensor([p[1] for p in pairs])

print(f"\n--- 训练 Skip-gram ---")
print(f"嵌入维度: {embed_dim}")

n_epochs = 200
for epoch in range(n_epochs):
    # 正样本得分
    pos_scores = model(center_words, context_words)
    pos_loss = -torch.log(torch.sigmoid(pos_scores) + 1e-10).mean()
    
    # 负采样：随机选择不是上下文的词作为负样本
    neg_words = torch.randint(0, vocab_size, (len(pairs),))
    neg_scores = model(center_words, neg_words)
    neg_loss = -torch.log(torch.sigmoid(-neg_scores) + 1e-10).mean()
    
    loss = pos_loss + neg_loss
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"  Epoch {epoch}: loss = {loss.item():.4f}")

print("\n✓ 训练完成")


3. 从零实现 Skip-gram Word2Vec
词表: {'and': 0, 'are': 1, 'cat': 2, 'chased': 3, 'dog': 4, 'friends': 5, 'log': 6, 'mat': 7, 'on': 8, 'sat': 9, 'the': 10}
词表大小: 11

训练样本数: 86
前5个样本:
  (the, cat)
  (the, sat)
  (cat, the)
  (cat, sat)
  (cat, on)

--- 训练 Skip-gram ---
嵌入维度: 10
  Epoch 0: loss = 2.2711
  Epoch 50: loss = 1.1906
  Epoch 100: loss = 1.0181
  Epoch 150: loss = 1.1254

✓ 训练完成


## 4. 词向量的性质


训练完成后，可以用余弦相似度比较词向量。语料很小时结果会不稳定，这是正常现象；Word2Vec 真正的效果来自大规模语料和大量共现统计。

经典类比 `king - man + woman ≈ queen` 的重点不是“向量会魔法”，而是向量空间中某些方向捕捉到了稳定语义关系。

In [5]:
print("\n" + "=" * 60)
print("4. 词向量的有趣性质")
print("=" * 60)

# 获取学到的词向量
embeddings = model.center_embeddings.weight.detach().numpy()

print("学到的词向量:")
for word, idx in sorted(word2idx.items()):
    print(f"  {word:8s}: {embeddings[idx][:5].round(3)}...")

# 计算相似度
print(f"\n--- 词语相似度 ---")
def word_similarity(word1, word2, embeddings, word2idx):
    v1 = embeddings[word2idx[word1]]
    v2 = embeddings[word2idx[word2]]
    return cosine_similarity(v1, v2)

word_pairs = [('cat', 'dog'), ('cat', 'mat'), ('dog', 'log'), 
              ('sat', 'chased'), ('the', 'on')]
for w1, w2 in word_pairs:
    sim = word_similarity(w1, w2, embeddings, word2idx)
    print(f"  cos({w1}, {w2}) = {sim:.4f}")

print("""
\n注意: 由于语料很小，结果可能不太理想。
在真实的大规模语料(如 Wikipedia)上训练，词向量会展现出令人惊奇的性质：

经典例子 (使用预训练的 Word2Vec):
  king - man + woman ≈ queen
  paris - france + japan ≈ tokyo
  
这说明词向量捕捉到了 语义关系 和 类比关系！
""")


4. 词向量的有趣性质
学到的词向量:
  and     : [-0.439  1.536 -0.648  1.093  0.877]...
  are     : [-0.722 -0.824  1.405  1.844  0.868]...
  cat     : [ 1.056 -1.033 -0.037 -0.67   1.092]...
  chased  : [ 0.728 -0.434 -1.152  1.337 -1.098]...
  dog     : [-0.252 -0.797  0.641  1.493 -0.5  ]...
  friends : [-0.    -0.021  0.667  0.339 -0.198]...
  log     : [-1.315 -0.128 -1.14   0.045 -1.175]...
  mat     : [-1.489 -0.875 -1.713  0.815  0.875]...
  on      : [ 1.281  0.522 -0.556 -1.419  0.076]...
  sat     : [ 1.309 -1.795 -0.633  0.717  0.371]...
  the     : [ 1.187 -0.816 -0.052  0.005 -2.119]...

--- 词语相似度 ---
  cos(cat, dog) = 0.2082
  cos(cat, mat) = 0.2788
  cos(dog, log) = 0.1089
  cos(sat, chased) = 0.6205
  cos(the, on) = 0.2827


注意: 由于语料很小，结果可能不太理想。
在真实的大规模语料(如 Wikipedia)上训练，词向量会展现出令人惊奇的性质：

经典例子 (使用预训练的 Word2Vec):
  king - man + woman ≈ queen
  paris - france + japan ≈ tokyo

这说明词向量捕捉到了 语义关系 和 类比关系！



## 5. 嵌入层在实际中的使用


`nn.Embedding` 是现代 NLP 模型的入口层。输入 token id，输出对应向量序列；后面的 RNN、CNN 或 Transformer 都在这些向量上继续计算。

重要细节：Embedding 的输入必须是整数索引，输出是浮点向量；训练时梯度会更新被查到的那些行。

In [6]:
print("\n" + "=" * 60)
print("5. 嵌入层在神经网络中的使用")
print("=" * 60)

print("""
【nn.Embedding 的本质】
nn.Embedding 其实就是一个查找表（lookup table）：
  - 内部存储一个 (vocab_size × embed_dim) 的矩阵
  - 给定词的索引，返回对应行的向量

等价于：对 one-hot 向量做矩阵乘法
  one_hot × W = embedding
  [0,0,1,0,0] × W(5×3) = W 的第 3 行

但直接查表比矩阵乘法快！

【在完整模型中】
文本 → 分词 → 词索引 → Embedding 层 → 向量序列 → RNN/Transformer → 输出
""")

# 演示 Embedding 层
embedding_layer = nn.Embedding(num_embeddings=10, embedding_dim=4)
print(f"Embedding 权重矩阵形状: {embedding_layer.weight.shape}")

# 查询
input_indices = torch.LongTensor([2, 5, 7])
output_vectors = embedding_layer(input_indices)
print(f"\n输入索引: {input_indices.tolist()}")
print(f"输出向量:\n{output_vectors.detach().numpy().round(4)}")

# 验证：直接索引等价
manual = embedding_layer.weight[2]
assert torch.allclose(output_vectors[0], manual)
print("\n✓ Embedding(2) == weight[2]，验证通过")

# 批量处理句子
sentences = torch.LongTensor([
    [1, 3, 5, 2],   # 句子1 (4个词)
    [4, 6, 8, 0],   # 句子2 (4个词)
])
embedded = embedding_layer(sentences)
print(f"\n批量嵌入:")
print(f"  输入: {list(sentences.shape)} (2个句子, 每句4个词)")
print(f"  输出: {list(embedded.shape)} (2个句子, 4个词, 4维向量)")

print("\n" + "=" * 60)
print("本节总结")
print("=" * 60)
print("""
关键要点：
1. One-Hot 高维稀疏，无法表达语义关系
2. 词嵌入：低维稠密向量，语义相近的词距离也近
3. Word2Vec 通过预测上下文来学习词向量
4. Skip-gram: 中心词→预测上下文词
5. nn.Embedding 就是一个可学习的查找表
6. 词嵌入是所有 NLP 模型的第一层

现代演进：
  Word2Vec (2013) → GloVe (2014) → ELMo (2018) → BERT (2018)
  
  区别：
  - Word2Vec/GloVe: 每个词一个固定向量
  - ELMo/BERT: 同一个词在不同语境中有不同向量（上下文化表示）
    例: "苹果很好吃" vs "苹果发布了新手机"
        → "苹果"在两句中的向量不同！

下一节：注意力机制 → Transformer 的核心
""")


5. 嵌入层在神经网络中的使用

【nn.Embedding 的本质】
nn.Embedding 其实就是一个查找表（lookup table）：
  - 内部存储一个 (vocab_size × embed_dim) 的矩阵
  - 给定词的索引，返回对应行的向量

等价于：对 one-hot 向量做矩阵乘法
  one_hot × W = embedding
  [0,0,1,0,0] × W(5×3) = W 的第 3 行

但直接查表比矩阵乘法快！

【在完整模型中】
文本 → 分词 → 词索引 → Embedding 层 → 向量序列 → RNN/Transformer → 输出

Embedding 权重矩阵形状: torch.Size([10, 4])

输入索引: [2, 5, 7]
输出向量:
[[ 0.8928 -0.0573  0.3463 -0.1465]
 [ 1.4318 -0.6638  1.0891 -0.0607]
 [-0.1101 -0.2885 -0.517   0.1115]]

✓ Embedding(2) == weight[2]，验证通过

批量嵌入:
  输入: [2, 4] (2个句子, 每句4个词)
  输出: [2, 4, 4] (2个句子, 4个词, 4维向量)

本节总结

关键要点：
1. One-Hot 高维稀疏，无法表达语义关系
2. 词嵌入：低维稠密向量，语义相近的词距离也近
3. Word2Vec 通过预测上下文来学习词向量
4. Skip-gram: 中心词→预测上下文词
5. nn.Embedding 就是一个可学习的查找表
6. 词嵌入是所有 NLP 模型的第一层

现代演进：
  Word2Vec (2013) → GloVe (2014) → ELMo (2018) → BERT (2018)

  区别：
  - Word2Vec/GloVe: 每个词一个固定向量
  - ELMo/BERT: 同一个词在不同语境中有不同向量（上下文化表示）
    例: "苹果很好吃" vs "苹果发布了新手机"
        → "苹果"在两句中的向量不同！

下一节：注意力机制 → Transformer 的核心



## 学习检查：词嵌入应该掌握什么

学完本节后，你应该能回答：

- 为什么 One-Hot 不能表达词与词之间的相似性？
- Skip-gram 中“中心词”和“上下文词”分别是什么？
- 负采样为什么能降低训练成本？
- `nn.Embedding(vocab_size, embed_dim)` 的权重矩阵形状是什么？
- 固定词向量和 BERT 这类上下文化词向量有什么区别？

如果这些问题能说清楚，你就已经具备理解 Transformer 输入层的基础。

## 常见误区

1. **词向量的每一维不一定能人工解释**：不要试图给每一维命名，重点看整体空间关系。
2. **小语料训练出的相似度可能不稳定**：这不是方法错了，而是统计信息不够。
3. **Embedding 不等于预训练词向量**：Embedding 是层；Word2Vec/GloVe 是训练词向量的方法；预训练词向量是训练好的参数。
4. **同一个词固定一个向量有局限**：`苹果` 在水果和公司两个语境中含义不同，这推动了 ELMo/BERT 等上下文化表示的发展。

## 深入理解：词向量到底学到了什么

词向量并不是人工设计出来的“猫=动物、苹果=水果”标签，而是模型在预测任务中自己学到的连续空间。训练时，模型会反复看到哪些词经常共同出现，哪些词几乎不共同出现。久而久之，相似上下文中的词会被推到相近位置。

从几何角度看，词嵌入把离散词表变成了一个向量空间。在这个空间里，可以用距离、角度、方向来描述词之间的关系。余弦相似度关注方向是否接近，因此常用于衡量语义相似度。

需要注意的是，词向量不是知识库。它不会显式保存“猫是一种动物”这样的三元组，而是通过大量上下文统计形成一种可被模型利用的表示。它的能力来自数据分布，局限也来自数据分布。

## Skip-gram 的训练直觉

Skip-gram 做的是一个很简单但有效的任务：给定中心词，判断哪些词更可能出现在它附近。

例如在句子 `the cat sat on the mat` 中，如果中心词是 `cat`，窗口大小为 2，那么上下文可能包括 `the`、`sat`。模型希望 `cat` 和这些上下文词点积更大，同时希望 `cat` 和随机采样的负样本词点积更小。

这件事可以理解成两种力：

- 正样本把真实共现的词向量拉近。
- 负样本把随机噪声词向量推远。

大量样本反复训练后，语义相近或用法相近的词会在空间中形成聚类。小语料中效果不明显，是因为共现统计太少；大语料中这种统计会稳定得多。

## 从词嵌入走向现代 LLM

Word2Vec 和 GloVe 学到的是静态词向量：同一个词无论出现在哪里，都对应同一个向量。这带来一个明显问题：多义词无法区分。

例如“苹果很好吃”和“苹果发布了新手机”中，“苹果”的含义不同。静态词向量只能给它一个折中表示。BERT、GPT 这类模型使用上下文化表示：词的最终向量会根据整句话动态变化。

现代 LLM 仍然有 embedding 层，但这个 embedding 只是起点。经过多层 Transformer 后，每个 token 的表示已经融合了上下文信息。理解静态词嵌入，可以帮助你理解 LLM 输入层；理解上下文化表示，则是理解 Transformer 的关键。

## 核心术语对照

- **Vocabulary**：词表，保存 token 到 id 的映射。
- **One-Hot**：只有一个位置为 1 的稀疏向量，简单但不含语义距离。
- **Embedding Matrix**：形状通常是 `(vocab_size, embed_dim)` 的可学习参数表。
- **Context Window**：中心词周围被用来构造训练样本的范围。
- **Negative Sampling**：用少量随机负样本近似完整词表分类，降低 Word2Vec 训练成本。

复习时可以把这些术语串起来：token 先通过词表变成 id，id 再通过 embedding matrix 查到向量，Skip-gram 利用 context window 和 negative sampling 训练这个向量空间。

## 课后练习：把词嵌入学扎实

建议你在 notebook 中尝试三个改动：第一，把语料中的 `cat/dog` 换成更多动物词和食物词，观察相似度是否更符合直觉；第二，把 `window_size` 从 2 改成 1 或 3，理解上下文范围如何影响训练样本；第三，把 `embed_dim` 改成 2，然后画出二维词向量散点图。

这些练习能帮助你把“词向量是学出来的空间”变成直观经验。不要只看最终相似度数字，也要观察训练数据如何构造，因为数据决定了模型能学到什么语义关系。

## 实战连接：从文本到模型输入的完整链路

在真实 NLP/LLM 系统中，词嵌入不是单独存在的。完整链路通常是：原始文本 → tokenizer → token id → embedding → 深层模型 → logits 或任务输出。

本节的 `nn.Embedding` 对应的是 token id 到向量的查表过程。Word2Vec 时代常以“词”为单位；现代 LLM 通常以 token 或子词为单位。也就是说，embedding 表里的每一行不一定对应一个完整中文词或英文词，可能对应一个字、子词、标点或特殊符号。

理解这一点很重要：当模型输出奇怪内容时，问题不一定出在 Transformer，也可能出在 tokenizer 如何切分文本、是否加入了特殊 token、是否截断了输入，或者 embedding 是否和模型权重匹配。

## 学习路线建议

建议按下面顺序巩固本节内容：

1. 先手算 One-Hot 的余弦相似度，确认所有不同词相似度为 0。
2. 再运行 Skip-gram 训练，观察 loss 是否下降。
3. 修改窗口大小，看看训练样本数量如何变化。
4. 修改 embedding 维度，观察小语料上相似度是否稳定。
5. 最后把 `nn.Embedding` 放到一个简单分类模型中，理解它如何作为第一层被训练。

掌握这些之后，再学习 tokenizer、Attention 和 Transformer 会顺很多，因为你已经知道离散 token 是如何进入连续向量空间的。